<a href="https://colab.research.google.com/github/aurorapiccione/bluesky-ai-network-analysis/blob/main/analisi_rete_bluesky.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install atproto networkx python-louvain matplotlib

from atproto import Client
import networkx as nx
from collections import defaultdict

client = Client()
client.login('tuo_handle.bsky.social', 'tua_password')
# ⚠️ consigliato: usa una "App Password" generata da Impostazioni > App Passwords,
# NON la password principale del tuo account

In [ ]:
# Cerca post recenti sul tema AI/tech
QUERY = "AI OR #AI OR #artificialintelligence OR #machinelearning"
posts = []

results = client.app.bsky.feed.search_posts({'q': QUERY, 'limit': 100})
posts.extend(results.posts)

# Se vuoi più dati, pagina con cursor:
cursor = results.cursor
while cursor and len(posts) < 500:  # regola il numero in base al tempo che hai
    results = client.app.bsky.feed.search_posts({'q': QUERY, 'limit': 100, 'cursor': cursor})
    posts.extend(results.posts)
    cursor = results.cursor

In [ ]:
G = nx.Graph()

for post in posts:
    author = post.author.handle
    uri = post.uri
    G.add_node(author)

    # Chi ha messo like a questo post
    try:
        likes = client.app.bsky.feed.get_likes({'uri': uri, 'limit': 100})
        for like in likes.likes:
            liker = like.actor.handle
            G.add_node(liker)
            if G.has_edge(author, liker):
                G[author][liker]['weight'] += 1
            else:
                G.add_edge(author, liker, weight=1)
    except Exception as e:
        print(f"Errore su {uri}: {e}")

In [ ]:
# Statistiche base
print("Nodi:", G.number_of_nodes())
print("Archi:", G.number_of_edges())
print("Densità:", nx.density(G))
print("Grado medio:", sum(dict(G.degree()).values()) / G.number_of_nodes())

# Distribuzione dei gradi (per verificare se è scale-free)
degrees = [d for n, d in G.degree()]
plt.hist(degrees, bins=30)
plt.xlabel("Grado"); plt.ylabel("Frequenza"); plt.title("Distribuzione dei gradi")
plt.show()

In [ ]:
import community as community_louvain  # python-louvain

partition = community_louvain.best_partition(G)
print("Numero di community trovate:", len(set(partition.values())))

# Visualizzazione colorata per community
pos = nx.spring_layout(G, k=0.3)
colors = [partition[n] for n in G.nodes()]
nx.draw(G, pos, node_color=colors, cmap=plt.cm.tab20, node_size=30, with_labels=False)
plt.show()